# SimDPS quick start

Fill a gap in your own recording with SimDPS and ACE-Step 1.5. In Colab, choose
*Runtime > Change runtime type > GPU* first.

In [ ]:
!pip install -q "simdps[acestep] @ git+https://github.com/s-turland/SimDPSInpainting"

Upload a recording, or leave `path = None` to use an example from librosa.

In [ ]:
path = None

try:
    from google.colab import files
    path = next(iter(files.upload()))
except (ImportError, StopIteration):
    pass

if path is None:
    import librosa
    path = librosa.ex("nutcracker")

In [ ]:
import simdps
from IPython.display import Audio, display

model = simdps.load("acestep")
audio, sr = simdps.load_audio(path)
print(f"{audio.shape[-1] / sr:.1f} s, {sr} Hz, {audio.shape[0]} channel(s)")

Choose the gap, in seconds. Two to five seconds is a good start.

In [ ]:
gap = (20.0, 23.0)
out = model.inpaint(audio, sr, gap, n_samples=2, seed=0)

a, b = out.gap
lo, hi = max(0, a - 5 * out.sample_rate), b + 5 * out.sample_rate
original = simdps.audio.resample(audio, sr, out.sample_rate)
print("original")
display(Audio(original[:, lo:hi].numpy(), rate=out.sample_rate))
for i, y in enumerate(out.audio):
    print(f"SimDPS {i + 1}")
    display(Audio(y[:, lo:hi].numpy(), rate=out.sample_rate))

Compare with the other methods: plain DPS, which has no candidates, and the
best search candidate on its own.

In [ ]:
for method in ("dps", "search", "solo"):
    y = model.inpaint(audio, sr, gap, method=method, seed=0).audio[0]
    print(method)
    display(Audio(y[:, lo:hi].numpy(), rate=out.sample_rate))

The candidates can also come from another recording, which pulls the gap
towards that material while keeping the context of the input.

In [ ]:
import librosa

other, _ = simdps.load_audio(librosa.ex("brahms"), sample_rate=sr)
y = model.inpaint(audio, sr, gap, source=other, method="solo", seed=0).audio[0]
display(Audio(y[:, lo:hi].numpy(), rate=out.sample_rate))

Save the result:

In [ ]:
simdps.save_audio("simdps.wav", out.audio[0], out.sample_rate)